# Rakshak AI — LoRA → Merged Model (Unsloth-based)

**Purpose**: Merge fine-tuned LoRA adapter into base Gemma 4, save as FP16, export to .litertlm.

This notebook uses Unsloth (same as training) to properly handle the custom
`Gemma4ClippableLinear` layers that PEFT alone cannot inject adapters into.

**Root cause of previous failure**: Loading the unsloth-patched base model with
raw `transformers` leaves Unsloth's custom layer classes in place.
`PeftModel.from_pretrained()` cannot create adapters for `Gemma4ClippableLinear`
because PEFT only knows standard types. The fix: load via `FastLanguageModel`
(exactly as training did) so PEFT sees compatible layers.

**Setup**:
1. Upload `rakshak-lora-final.zip` as Kaggle Dataset named `rakshak-lora-final`
2. Enable GPU T4 x2 + Internet

**Output**:
- `rakshak-merged-16bit/` — merged model (sharded SafeTensors, pure FP16)
- `rakshak-ai.litertlm` — LiteRT model for on-device flutter_gemma

In [ ]:
# Cell 1: Install dependencies
#
# Issues:
# 1. unsloth upgrades torch 2.10.0->2.12.1 but torchvision 0.25.0 stays -> RuntimeError
# 2. transformers 5.x has NotImplementedError in weight conversion revert for Gemma 4
#
# Fixes:
# - Don't pin transformers (4.57.6 lacks gemma4 support); let unsloth pull in 5.x
# - Upgrade torchvision with -U to fix version mismatch
# - Monkey-patch revert_weight_conversion before save (Cell 4)
import subprocess, sys, os, gc, torch, json, shutil, zipfile

print("Installing Unsloth + pinned deps...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git",
    "bitsandbytes", "peft", "accelerate"])
print("Fixing torch/torchvision version mismatch...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
    "torchvision", "torchaudio"])
print("Done\n")

from unsloth import FastModel
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ── Paths ──
BASE_MODEL_ID = "unsloth/gemma-4-e2b-it-unsloth-bnb-4bit"
LORA_PATH = "/kaggle/input/datasets/abhishekguptaagp/rakshak-lora-final"
OUTPUT_DIR = "/kaggle/working/rakshak-converted"
MERGED_PATH = "/kaggle/working/rakshak-merged-16bit"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MERGED_PATH, exist_ok=True)

print(f"Base model: {BASE_MODEL_ID}")
print(f"LoRA: {LORA_PATH}")

lora_files = os.listdir(LORA_PATH)
print(f"LoRA files: {len(lora_files)}")

In [ ]:
# Cell 2: Load base model via Unsloth
print("=" * 60)
print("Loading Gemma 4 E2B IT via Unsloth...")
print("=" * 60)

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    device_map="auto",
)

print(f"Model loaded: {type(model).__name__}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 3: Load LoRA adapter
print("=" * 60)
print("Loading LoRA adapter...")
print("=" * 60)

with open(os.path.join(LORA_PATH, "adapter_config.json")) as f:
    cfg = json.load(f)
print(f"LoRA rank: {cfg.get('r', '?')}, alpha: {cfg.get('lora_alpha', '?')}")

model = PeftModel.from_pretrained(model, LORA_PATH)
print("LoRA loaded successfully")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 4: Merge LoRA + dequantize 4-bit -> FP16 via manual save
print("=" * 60)
print("Merging + saving as FP16...")
print("=" * 60)

# transformers 5.x revert_weight_conversion has NotImplementedError for Gemma 4.
# save_pretrained_merged is broken. We do it manually instead:
#   1) merge_and_unload() — merges LoRA into base weights
#   2) iterate modules, dequantize 4-bit layers via bitsandbytes
#   3) save FP16 state_dict + config + tokenizer with safetensors
import bitsandbytes as bnb
import safetensors.torch
from transformers.modeling_utils import shard_checkpoint

# ── Step A: merge LoRA, get clean base model ──
model = model.merge_and_unload()
print("LoRA merged")

# Inspect current state
sd = model.state_dict()
first_key = next(iter(sd))
first_val = sd[first_key]
print(f"Sample key: {first_key}, dtype: {first_val.dtype}")

# Check if any keys still have PEFT structure (indicating merge failed)
if any(k.endswith('.base_layer.weight') or '.lora_' in k for k in sd):
    print("ERROR: LoRA merge incomplete — key format still expanded.")
    print("Falling back to save_pretrained_merged in a subprocess...")
    raise RuntimeError("merge_and_unload did not collapse LoRA keys")

# Build FP16 state dict: dequantize 4-bit layers, cast the rest
sd = {}
dequantized_count = 0
for module_name, module in model.named_modules():
    if not module_name:
        continue
    if hasattr(module, 'weight') and module.weight is not None:
        w = module.weight
        qs = getattr(module, 'quant_state', None)
        if w.dtype == torch.uint8 and qs is not None:
            w = bnb.functional.dequantize_4bit(w, qs)
            dequantized_count += 1
        sd[module_name + '.weight'] = w.to(torch.float16).contiguous()
    if hasattr(module, 'bias') and module.bias is not None:
        sd[module_name + '.bias'] = module.bias.to(torch.float16).contiguous()

print(f"Dequantized {dequantized_count} layers, {len(sd)} total tensors")

# ── Step B: save config + tokenizer ──
os.makedirs(MERGED_PATH, exist_ok=True)
model.config.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)
print("Config + tokenizer saved")

# ── Step C: write sharded safetensors ──
shards, index = shard_checkpoint(sd, max_shard_size="2GB")
if index is not None:
    for shard_file, shard_data in shards.items():
        safetensors.torch.save_file(shard_data, os.path.join(MERGED_PATH, shard_file))
    with open(os.path.join(MERGED_PATH, "model.safetensors.index.json"), "w") as f:
        json.dump(index, f, indent=2)
else:
    safetensors.torch.save_file(sd, os.path.join(MERGED_PATH, "model.safetensors"))

total_bytes = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fn in os.walk(MERGED_PATH) for f in fn
)
print(f"Saved. Size: {total_bytes / 1024**3:.2f} GB")

# ── Step D: verify + inference test (load with FastModel, 4-bit to save VRAM) ──
print("\nVerifying with FastModel + inference test...")
del model
gc.collect()
torch.cuda.empty_cache()

vm, vtokenizer = FastModel.from_pretrained(
    model_name=MERGED_PATH,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    device_map="auto",
)

test_input = "Patient not breathing. Triage category?"
messages = [{"role": "user", "content": [{"type": "text", "text": test_input}]}]
inputs = vtokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(vm.device)

outputs = vm.generate(input_ids=inputs, max_new_tokens=64, do_sample=False)
response = vtokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(f"Input: {test_input}")
print(f"Output: {response}")
print("Verification passed")

del vm, vtokenizer, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 5: Export to .litertlm
print("=" * 60)
print("Exporting to .litertlm...")
print("=" * 60)

litertlm_path = "/kaggle/working/rakshak-ai.litertlm"

try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ai-edge-torch"])
    import ai_edge_torch
    from ai_edge_torch.generative.utilities import export as ai_edge_export

    print("Running ai-edge-torch export...")
    ai_edge_export.model_to_litert(
        MERGED_PATH,
        output_path=litertlm_path,
        tokenizer=tokenizer,
        seq_length=2048,
    )
    size = os.path.getsize(litertlm_path) / 1024**3
    print(f"Exported: {litertlm_path} ({size:.2f} GB)")
except ImportError:
    print("ai-edge-torch unavailable. Merged model saved for offline conversion.")
except Exception as e:
    print(f"Export failed: {e}")

In [ ]:
# Cell 6: Summary
print("=" * 60)
print("CONVERSION COMPLETE")
print("=" * 60)

print("\nOutput:")
for root, _, files in os.walk("/kaggle/working"):
    for fn in sorted(files):
        fp = os.path.join(root, fn)
        size_mb = os.path.getsize(fp) / 1024**2
        if size_mb > 1:
            print(f"  {fn}: {size_mb:.1f} MB")

print("\nDeploy:")
print("  adb push rakshak-ai.litertlm /sdcard/Download/")